In [ ]:
!pip install torchvision


In [ ]:
import torch 
import os   # to read files (structure their path)
from torch.utils.data import DataLoader,Dataset
from torchvision import transforms      # to transform different images (crop / convert into tensor etc.)
from PIL import Image    # Python Imaging Library

In [ ]:
# images load => transform => dataset of all images
class ImageProcessor:
    def __init__(self,root_dir_path,transformations=None):
        self.root_dir_path = root_dir_path
        self.transformations = transformations

        # list of path for all images
        self.all_img_paths = [os.path.join(root_dir_path,img) for img in os.listdir(root_dir_path)]

    def __len__(self):
        return len(self.all_img_paths)

    def __getitem__(self,idx):
        img_path=self.all_img_paths[idx]
        img=Image.open(img_path).convert("RGB")

        if self.transformations:
            img = self.transformations(img)

        return img
    

In [ ]:
root_dir_path="./img_align_celeba/img_align_celeba"

transformations=transforms.Compose([
    transforms.CenterCrop(178),    # we want to crop our image to square form (178 x 218 => 178 x 178)
    transforms.Resize(64), # 64x64
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)) # in the image 64x64 converting each pixel into the range [-1,1]
])


In [ ]:
dataset = ImageProcessor(root_dir_path, transformations)
print(f"loaded {len(dataset)} images")

In [ ]:
dataloader=DataLoader(dataset,batch_size=128,shuffle=True)

## Generator Network

In [ ]:
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [ ]:
class Generator(nn.Module):
    def __init__(self, z_dim=100, img_channels=3): # 3 is for RGB
        super(Generator,self).__init__()

    # fully connected dense layers 
        self.model=nn.Sequential(
            nn.Linear(z_dim, 256), # 100 => 256 (upsampling)
            nn.ReLU(),
    
            nn.Linear(256, 512), # 256 => 512
            nn.ReLU(),
    
            nn.Linear(512, 1024), # 512 => 1024
            nn.ReLU(),
    
            nn.Linear(1024, 64*64*img_channels), 
            nn.Tanh()  # Tanh generates values in the range [-1,1] which should match with the normalized dimensions of pixel
        )

    def forward(self, z):
        img=self.model(z)
        img=img.view(img.size(0), 3, 64, 64)  # img.size(0) => batch size; image dimension => 64x64x3
        return img
    
    
        # fake img => 64 x 64 x 3 x batch_size

## Discriminator

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, img_channels=3): # 3 is for RGB
        super(Discriminator,self).__init__()
    
        # fully connected dense layers 
        self.model=nn.Sequential(
            nn.Flatten(),    # 4D tensor => 1D
            
            nn.Linear(img_channels*64*64, 1024), 
            nn.LeakyReLU(0.2, inplace=True),
    
            nn.Linear(1024, 512), 
            nn.LeakyReLU(0.2, inplace=True),
    
            nn.Linear(512, 256), 
            nn.LeakyReLU(0.2, inplace=True),
    
            nn.Linear(256,1), 
            nn.Sigmoid()  # probability of being real/fake
        )

    def forward(self, img):
        return self.model(img)


In [ ]:
GAN_loss=nn.BCELoss()

generator=Generator()
g_optimizer=optim.Adam(generator.parameters(), lr=0.002, betas=(0.5,0.999))

discriminator=Discriminator()
d_optimizer=optim.Adam(generator.parameters(), lr=0.002, betas=(0.5,0.999))

In [ ]:
import torch
# device
if torch.backends.mps.is_available():
    device=torch.device("mps")

elif torch.cuda.is_available():
    device=torch.device("cuda")

else:
    device=torch.device("cpu")

print(f"device is {device}")

In [ ]:
generator=generator.to(device)
discriminator=discriminator.to(device)

## Training the GAN

In [ ]:
def train(generator,discriminator,dataloader,epochs=10):
    for epoch in range(epochs):
        for i, imgs in enumerate(dataloader):
            real_imgs=imgs.to(device)
            batch_size=real_imgs.size(0)

            # create real images labels and fake images labels
            real_labels=torch.ones(batch_size,1).to(device)    # [1, 1, 1, .......]
            fake_labels=torch.zeros(batch_size,1).to(device)   # [0, 0, 0, .......]

            # Train the discriminator
            d_optimizer.zero_grad()        # we want to reset all the gradients in this particular batch

            fake_imgs = generator(torch.randn(batch_size,100).to(device))      # we want to pass some random noise inside the generator

            real_loss = GAN_loss(discriminator(real_imgs),real_labels)
            fake_loss = GAN_loss(discriminator(fake_imgs.detach()),fake_labels) # we have to pass the fake_imgs by detaching => fake data will not propagate into the discriminator and does not make changes in it

            d_loss = (real_loss + fake_loss)/2

            d_loss.backward()   # backward propagation
            d_optimizer.step()  # updation of gradients

            # Train the generator
            g_optimizer.zero_grad()

            g_loss=GAN_loss(discriminator(fake_imgs), real_labels)

            g_loss.backward()
            g_optimizer.step()

            if i % 50 == 0:
                print(f" for epoch: {epoch+1}/{epochs}... batch: {i+1}... G-loss: {g_loss}... Discriminator loss: {d_loss}")
                
        # save generated images for each epoch
        save_generated_images(generator,epoch,device)

In [ ]:
import matplotlib.pyplot as plt
import torchvision

def save_generated_images(generator, epoch, device, num_imgs=8):

    z=torch.randn(num_imgs,100).to(device)
    
    fake_imgs = generator(z).detach().cpu()

    # [-1, 1] => [0, 1]
    grid=torchvision.utils.make_grid(generated_imgs, nrow=4, Normalize=True)

    plt.imshow(np.transpose(grid,(1,2,0)))
    plt.title(f"epoch {epoch+1}")
    plt.axis("off")
    plt.show()

In [ ]:
train(generator, discriminator, dataloader, epochs=5)